# Musicology prompt-intervention eval — Colab runner

Runs the WeaveMuse manager agent over `data/eval/tasks_musicology.jsonl` under the
`default` vs `expert` prompt variants, captures traces, and scores them with an LLM judge.
See `weavemuse/eval/README.md` and `weavemuse/eval/STUDY_LOG.md`.

**Agent set for this sweep:** the manager keeps `musicology_analysis_agent`, `chat_musician`,
and `web_search_agent`. The generative and audio agents (`symbolic_music_agent`,
`audio_generation_agent`, `audio_analysis_agent`) are excluded via `--exclude-agents` — the
tasks are score/metadata analysis, no generation or audio input.

**Before you start:**
1. **Runtime → Change runtime type → GPU.** `chat_musician` is a local model, so a GPU is
   required. T4 (16 GB) works with a smaller `--model-id`; the 30B default needs A100/L4.
   If `nvidia-smi` says *command not found*, you are on a CPU runtime.
2. The repo is **private** — the clone cell asks for a GitHub PAT with read access to
   `noamsprei/LAFS-weavemuse`.
3. **Confidentiality:** the Didone CSVs are not in the repo; you place them on this VM
   yourself (Drive mount). That puts the data on Google infrastructure for the session.

In [ ]:
# Expect a GPU line here. 'command not found' => switch to a GPU runtime and rerun.
!nvidia-smi || echo 'NO GPU RUNTIME — Runtime > Change runtime type > GPU'

In [ ]:
# Clone the private PR branch. Paste a GitHub PAT with read access to the repo.
import os
from getpass import getpass

_pat = getpass("GitHub PAT: ").strip()
_url = f"https://{_pat}@github.com/noamsprei/LAFS-weavemuse.git"
!git clone --branch eval-harness --single-branch "$_url" LAFS-weavemuse
del _pat, _url
%cd /content/LAFS-weavemuse
!git log --oneline -3

In [ ]:
# Install. torch is already present on Colab; this pulls smolagents, pandas,
# the remote tool clients, and the judge's anthropic/openai clients.
!pip install -q -e ".[remote]"
print("\nif pip printed a 'restart runtime' notice: Runtime > Restart, then rerun from the %cd cell")

In [ ]:
from getpass import getpass

# HF_TOKEN is optional for this sweep (generative/audio agents are excluded); it is
# only used for gated backbone downloads. Press Enter to skip.
_hf = getpass("HF token (optional, Enter to skip): ").strip()
if _hf:
    os.environ["HF_TOKEN"] = _hf

# Anthropic key: only the remote judge uses it. Blank => judge with the local model.
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (remote judge; blank = local): ").strip()

## Didone data

Point `DIDONE_DATA_DIR` at a folder containing:
```
metadata.csv
harmony_analysis/tonal_plan_overview.csv
harmony_analysis/tonal_plan_segments.csv
harmony_analysis/parsed_harmony_events.csv
text_tonal_alignment/section_tonal_plan.csv
textual_plan/textual_plan_overview.csv
textual_plan/textual_plan_sections.csv
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# adjust to where you put the data in your Drive
os.environ["DIDONE_DATA_DIR"] = "/content/drive/MyDrive/didone_data"

# eval outputs (traces, manifests, scores) write straight here -- survives a
# runtime disconnect, and you can open them in Drive while a sweep runs.
OUT_ROOT = "/content/drive/MyDrive/musicology_eval"
os.makedirs(OUT_ROOT, exist_ok=True)

!ls -la "$DIDONE_DATA_DIR" && echo '---' && ls "$DIDONE_DATA_DIR"/harmony_analysis

In [ ]:
# Sanity check: tools load and return data for a sample aria
from weavemuse.tools.didone_tools import didone_tools

t = {x.name: x for x in didone_tools()}
print(t["get_tonal_plan"].forward("0012")[:1500])

# Compute domain-expert reference answers (sw1/sw2/sw4/mw1) into an augmented,
# gitignored dataset the judge scores task_success against.
!python scripts/build_references.py
TASKS = "data/eval/tasks_musicology.local.jsonl"

## Smoke test — 1 task, both variants

Run one task under `default` and `expert`, stream the agent's steps live, then render
both saved traces side by side. Check the traces look sane before the full sweep.

In [ ]:
SMOKE_TASK = "sw1_modulation__0012"
RUN_ID = "smoke5"

# --output-dir -> Drive (traces land there per-pair as they finish).
# --overwrite: always redo the smoke (you're iterating on prompts). The core
# sweep below does NOT overwrite -- it resumes from existing traces.
!python scripts/run_eval.py \
  --dataset {TASKS} \
  --variants data/eval/variants_musicology.json \
  --run-id {RUN_ID} \
  --output-dir {OUT_ROOT} \
  --overwrite \
  --tool-mode hybrid \
  --exclude-agents symbolic_music_agent,audio_generation_agent,audio_analysis_agent \
  --max-steps 16 \
  --task-ids {SMOKE_TASK}
# T4 only (weak): add  --model-id Qwen/Qwen2.5-7B-Instruct   (A100/L4 auto-picks 14B; 32B won't fit Colab disk)

In [ ]:
import json, textwrap
from pathlib import Path

def render_trace(run_id, task_id, variant):
    p = Path(f"{OUT_ROOT}/{run_id}/traces/{task_id}/{variant}.json")
    if not p.exists():
        print(f"[missing] {p}"); return
    tr = json.loads(p.read_text())
    print("=" * 100)
    print(f"{task_id}  ·  variant={variant}  ·  state={tr['state']}")
    tk, tm = tr.get("token_usage"), tr.get("timing")
    if tk: print(f"tokens: in={tk.get('input_tokens')} out={tk.get('output_tokens')}", end="  ")
    if tm: print(f"wall={tm.get('duration'):.0f}s" if tm.get('duration') else "")
    if tr.get("error"): print(f"ERROR: {tr['error']}")
    print("-" * 100)
    for m in tr.get("messages", []):
        role = m.get("role", "?")
        content = m.get("content", "")
        if isinstance(content, list):
            content = "\n".join(c.get("text", str(c)) if isinstance(c, dict) else str(c) for c in content)
        content = str(content).strip()
        if not content:
            continue
        print(f"\n### {role} ###")
        print(textwrap.shorten(content, width=4000, placeholder=" …[truncated]") if len(content) > 4000 else content)
    print("\n" + "-" * 100)
    print("FINAL ANSWER:\n" + str(tr.get("output")))
    print("=" * 100 + "\n")

for v in ("default", "expert"):
    render_trace(RUN_ID, SMOKE_TASK, v)

## Full core sweep

Only after the smoke traces look sane. 33 `core` tasks (SW1 + SW2 + MW1) × 2 variants = 66 runs.
Long — on an A100/14B expect a few hours.

**Resume:** each finished trace is written to Drive immediately and the manifest is checkpointed
after every pair. If the runtime dies, just re-run this cell — it skips pairs that already have a
trace and continues. To force a full redo, add `--overwrite`. Drop `--task-ids` for all 56 tasks.

In [ ]:
import json

RUN_ID = "colab_core"
core = ",".join(json.loads(l)["task_id"] for l in open(TASKS)
                if "core" in json.loads(l)["tags"])

!python scripts/run_eval.py \
  --dataset {TASKS} \
  --variants data/eval/variants_musicology.json \
  --run-id {RUN_ID} \
  --output-dir {OUT_ROOT} \
  --tool-mode hybrid \
  --exclude-agents symbolic_music_agent,audio_generation_agent,audio_analysis_agent \
  --max-steps 16 \
  --task-ids {core}
# for ALL 56 tasks: delete the two lines above that build/pass --task-ids

In [ ]:
# Score the trace(s) for whichever RUN_ID was last run. Scores write to {OUT_ROOT}/{RUN_ID}/scores/.
# --dataset feeds the judge the computed reference answers (sw1/sw2/sw4/mw1).
# remote => needs ANTHROPIC_API_KEY; local => same local model (smoke only, biased).
import subprocess
backend = "remote" if os.environ.get("ANTHROPIC_API_KEY") else "local"
subprocess.run([
    "python", "scripts/run_judge.py",
    "--traces-dir", f"{OUT_ROOT}/{RUN_ID}/traces",
    "--dataset", TASKS,
    "--backend", backend,
    "--rubric", "weavemuse/eval/rubrics/default.json",
], check=True)

## Next

Everything is on Drive under `MyDrive/musicology_eval/<run_id>/` — `traces/`, `manifest.json`,
and after judging `scores/`. Nothing to copy manually.

- Read the rendered smoke traces above: does the sub-agent call a tool before answering, apply
  the *stated* counting rule (not just match field names), and name real key areas?
- If sane: run the **Full core sweep** cell, then the judge cell.
- Hand-grade ~5 final answers as an expert; check the judge's `scores/**/*.judge.json` agree.
- Then the full 56-task sweep, and (once it exists) `scripts/summarize_eval.py` to pivot scores.